In [0]:
"""
id: aggregate_9
template: aggregate
templateVersion: 1.0.0
name: aggregate_9
position:
  x: 1200
  y: 77.5
description:
  text: Group data by city and compute total order count and maximum unit price for each city.
  hash: 962dd82f
previewCodeHash: c30045ac2c55c035
previewMode: "1000"
config:
  group_bys:
    - expr: city
      type: column
  aggregations:
    - columnExpr:
        expr: order_id
        type: column
      fn: COUNT
      alias: Total_count
    - columnExpr:
        expr: unit_price
        type: column
      fn: MAX
      alias: MaxAmount
input:
  - node: join_8
    input_port: data
    output_port: joined_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "city",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "column"
            },
            "fn": "COUNT",
            "alias": "Total_count",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "unit_price",
                "type": "column"
            },
            "fn": "MAX",
            "alias": "MaxAmount",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["join_8.joined_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_9.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: python_5
template: python
templateVersion: 1.0.0
name: API_Customers
position:
  x: 300
  y: 155
description:
  text: Load customer data from a CSV file online or use provided input data.
  hash: 257c4cdf
previewCodeHash: 527a04700467bd96
previewMode: "1000"
config:
  code: |
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd 
        import requests
        from io import StringIO
        url = "https://raw.githubusercontent.com/vilasagaram-saishiva/LakeFlow-Designer--Databricks--Basic/refs/heads/main/customers.csv"
        response = requests.get(url)
        csv_data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd 
        import requests
        from io import StringIO
        url = "https://raw.githubusercontent.com/vilasagaram-saishiva/LakeFlow-Designer--Databricks--Basic/refs/heads/main/customers.csv"
        response = requests.get(url)
        csv_data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_5.result"] = out["result"]

In [0]:
"""
id: python_6
template: python
templateVersion: 1.0.0
name: API_ Shipments
position:
  x: 600
  y: 310
description:
  text: Load shipment data from a CSV file if no input data is provided.
  hash: b440382b
previewCodeHash: 3bba1dc69aad47f3
previewMode: "1000"
config:
  code: |
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd 
        import requests
        from io import StringIO
        url = "https://raw.githubusercontent.com/vilasagaram-saishiva/LakeFlow-Designer--Databricks--Basic/refs/heads/main/shipments.csv"
        response = requests.get(url)
        csv_data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd 
        import requests
        from io import StringIO
        url = "https://raw.githubusercontent.com/vilasagaram-saishiva/LakeFlow-Designer--Databricks--Basic/refs/heads/main/shipments.csv"
        response = requests.get(url)
        csv_data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_6.result"] = out["result"]

In [0]:
"""
id: source_1
template: source
templateVersion: 2.0.0
name: orders
position:
  x: 0
  y: 77.5
description:
  text: Retrieve all data from the orders table.
  hash: 71e002fa
previewCodeHash: d5d2411d97bd5608
previewMode: "1000"
config:
  table_source:
    tableName: desginer_catalog.raw.orders
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "desginer_catalog.raw.orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_1.data"] = out["data"]

In [0]:
"""
id: source_3
template: source
templateVersion: 2.0.0
name: order_items
position:
  x: 0
  y: 232.5
description:
  text: Load all data from the table 'desginer_catalog.raw.order_items'.
  hash: cfe99e5d
previewCodeHash: 6e57320a0759d300
previewMode: "1000"
config:
  table_source:
    tableName: desginer_catalog.raw.order_items
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "desginer_catalog.raw.order_items"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_3.data"] = out["data"]

In [0]:
"""
id: source_4
template: source
templateVersion: 2.0.0
name: reviews
position:
  x: 0
  y: 465
description:
  text: Read all data from the reviews table.
  hash: e22061ce
previewCodeHash: b83c3dfa98901f47
previewMode: "1000"
config:
  table_source:
    tableName: desginer_catalog.raw.reviews
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "desginer_catalog.raw.reviews"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_4.data"] = out["data"]

In [0]:
"""
id: join_4
template: join
templateVersion: 1.0.0
name: Order_joins_OrderItems
position:
  x: 300
  y: 0
description:
  text: Keep all records from the left table and add matching records from the right table based on order_id.
  hash: 84eea7d3
previewCodeHash: fb37371b4dfd57b3
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.order_id = right.order_id
  expressions:
    - "`left`.order_id"
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`right`.order_item_id"
    - "`right`.product_id"
    - "`right`.quantity"
    - "`right`.unit_price"
    - "`right`.discount_pct"
    - "`right`.line_total"
input:
  - node: source_1
    input_port: left
    output_port: data
  - node: source_3
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.order_id = right.order_id",
    "expressions": [
        "`left`.order_id",
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`right`.order_item_id",
        "`right`.product_id",
        "`right`.quantity",
        "`right`.unit_price",
        "`right`.discount_pct",
        "`right`.line_total"
    ]
}
inputs = {
    "left": ctx["source_1.data"],
    "right": ctx["source_3.data"]
}
out = run(config, inputs, spark)
ctx["join_4.joined_data"] = out["joined_data"]

In [0]:
"""
id: ai_function_16
template: ai_function
templateVersion: 1.0.0
name: Sentimental Analysis
position:
  x: 300
  y: 465
description:
  text: Add sentiment analysis scores for each review.
  hash: 99865d2e
previewCodeHash: f0f7ceb20547c960
previewMode: "1000"
config:
  expressions:
    - ai_analyze_sentiment(review_body)
input:
  - node: source_4
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"ai_data": df}

    return {"ai_data": df.selectExpr(*expressions, "*")}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_analyze_sentiment(review_body)"
    ]
}
inputs = {
    "data": ctx["source_4.data"]
}
out = run(config, inputs, spark)
ctx["ai_function_16.ai_data"] = out["ai_data"]

In [0]:
"""
id: join_7
template: join
templateVersion: 1.0.0
name: order&Item_join_Cust
position:
  x: 600
  y: 77.5
description:
  text: Left join two tables on customer_id and keep specified columns from both sides.
  hash: 3eccef0a
previewCodeHash: 3e549a48b88f00ea
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.customer_id = right.customer_id
  expressions:
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`left`.order_item_id"
    - "`left`.product_id"
    - "`left`.quantity"
    - "`left`.unit_price"
    - "`left`.discount_pct"
    - "`left`.line_total"
    - "`left`.order_id"
    - "`right`.customer_name"
    - "`right`.email"
    - "`right`.phone"
    - "`right`.city"
    - "`right`.state"
    - "`right`.country"
    - "`right`.created_date"
input:
  - node: python_5
    input_port: right
    output_port: result
  - node: join_4
    input_port: left
    output_port: joined_data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.customer_id = right.customer_id",
    "expressions": [
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`left`.order_item_id",
        "`left`.product_id",
        "`left`.quantity",
        "`left`.unit_price",
        "`left`.discount_pct",
        "`left`.line_total",
        "`left`.order_id",
        "`right`.customer_name",
        "`right`.email",
        "`right`.phone",
        "`right`.city",
        "`right`.state",
        "`right`.country",
        "`right`.created_date"
    ]
}
inputs = {
    "right": ctx["python_5.result"],
    "left": ctx["join_4.joined_data"]
}
out = run(config, inputs, spark)
ctx["join_7.joined_data"] = out["joined_data"]

In [0]:
"""
id: workspace.default.output_17
template: output
templateVersion: 1.0.0
name: desginer_catalog.enriched.Sentimental_table
position:
  x: 600
  y: 465
description:
  text: Save data to a table, replacing existing content.
  hash: 5f5bf8c0
previewMode: "1000"
config:
  catalog: desginer_catalog
  schema: enriched
  table_name: Sentimental_table
input:
  - node: ai_function_16
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "desginer_catalog",
    "schema": "enriched",
    "table_name": "Sentimental_table"
}
inputs = {
    "data": ctx["ai_function_16.ai_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: join_8
template: join
templateVersion: 1.0.0
name: One_Big_Table
position:
  x: 900
  y: 155
description:
  text: Left join on order_id, keeping selected columns from both inputs.
  hash: 148795a4
previewCodeHash: 496cc9626f683d94
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.order_id = right.order_id
  expressions:
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`left`.order_item_id"
    - "`left`.product_id"
    - "`left`.quantity"
    - "`left`.unit_price"
    - "`left`.discount_pct"
    - "`left`.line_total"
    - "`left`.order_id"
    - "`left`.customer_name"
    - "`left`.email"
    - "`left`.phone"
    - "`left`.city"
    - "`left`.state"
    - "`left`.country"
    - "`left`.created_date"
    - "`right`.shipment_id"
    - "`right`.ship_date"
    - "`right`.ship_mode"
    - "`right`.shipping_cost"
input:
  - node: join_7
    input_port: left
    output_port: joined_data
  - node: python_6
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.order_id = right.order_id",
    "expressions": [
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`left`.order_item_id",
        "`left`.product_id",
        "`left`.quantity",
        "`left`.unit_price",
        "`left`.discount_pct",
        "`left`.line_total",
        "`left`.order_id",
        "`left`.customer_name",
        "`left`.email",
        "`left`.phone",
        "`left`.city",
        "`left`.state",
        "`left`.country",
        "`left`.created_date",
        "`right`.shipment_id",
        "`right`.ship_date",
        "`right`.ship_mode",
        "`right`.shipping_cost"
    ]
}
inputs = {
    "left": ctx["join_7.joined_data"],
    "right": ctx["python_6.result"]
}
out = run(config, inputs, spark)
ctx["join_8.joined_data"] = out["joined_data"]

In [0]:
"""
id: transform_month
template: transform
templateVersion: 1.0.0
name: Add month column
position:
  x: 1200
  y: 232.5
description:
  text: Keep order date, order status, and add order month from order date.
  hash: d45d5419
previewMode: "1000"
config:
  expressions:
    - "`left`.order_date"
    - "`left`.order_status"
    - EXTRACT(MONTH FROM left.order_date) AS `order_month`
input:
  - node: join_8
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "`left`.order_date",
        "`left`.order_status",
        "EXTRACT(MONTH FROM left.order_date) AS `order_month`"
    ]
}
inputs = {
    "data": ctx["join_8.joined_data"]
}
out = run(config, inputs, spark)
ctx["transform_month.transformed_data"] = out["transformed_data"]

In [0]:
"""
id: sort_10
template: sort
templateVersion: 1.0.0
name: sort_10
position:
  x: 1500
  y: 77.5
description:
  text: Sort by MaxAmount in descending order.
  hash: ae09c974
previewCodeHash: c31c1686ea3c7f5a
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: MaxAmount
        type: column
      sortBy: DESC
input:
  - node: aggregate_9
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "MaxAmount",
                "type": "column"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_9.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_10.sorted_data"] = out["sorted_data"]

In [0]:
"""
id: aggregate_order_status_month
template: aggregate
templateVersion: 1.0.0
name: Count order_status per month
position:
  x: 1500
  y: 232.5
description:
  text: Count order status grouped by order month and order status.
  hash: cb6fa602
previewCodeHash: b2278f993ee7f5bf
previewMode: "1000"
config:
  group_bys:
    - expr: order_month
      type: expr
    - expr: left.order_status
      type: expr
  aggregations:
    - columnExpr:
        expr: left.order_status
        type: expr
      fn: COUNT
      alias: status_count
input:
  - node: transform_month
    input_port: data
    output_port: transformed_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "order_month",
            "type": "expr"
        },
        {
            "expr": "left.order_status",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "left.order_status",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "status_count",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["transform_month.transformed_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_order_status_month.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: sort_order_status_month
template: sort
templateVersion: 1.0.0
name: Sort by month DESC
position:
  x: 1800
  y: 232.5
description:
  text: Sort data by order_month in descending order.
  hash: fc3a5dae
previewCodeHash: c9abc05382aa0859
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: order_month
        type: expr
      sortBy: DESC
input:
  - node: aggregate_order_status_month
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "order_month",
                "type": "expr"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_order_status_month.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_order_status_month.sorted_data"] = out["sorted_data"]

In [0]:
"""
id: workspace.default.output_11
template: output
templateVersion: 1.0.0
name: desginer_catalog.enriched.output_11
position:
  x: 1800
  y: 77.5
description:
  text: Save data as a new table, replacing existing one if present.
  hash: e17103d1
previewMode: "1000"
config:
  catalog: desginer_catalog
  schema: enriched
  table_name: output_11
input:
  - node: sort_10
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "desginer_catalog",
    "schema": "enriched",
    "table_name": "output_11"
}
inputs = {
    "data": ctx["sort_10.sorted_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: workspace.default.output_15
template: output
templateVersion: 1.0.0
name: desginer_catalog.enriched.OrderStatus
position:
  x: 2100
  y: 232.5
description:
  text: Replace the specified table with new data.
  hash: 4a81b372
previewMode: "1000"
config:
  catalog: desginer_catalog
  schema: enriched
  table_name: OrderStatus
input:
  - node: sort_order_status_month
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "desginer_catalog",
    "schema": "enriched",
    "table_name": "OrderStatus"
}
inputs = {
    "data": ctx["sort_order_status_month.sorted_data"]
}
out = run(config, inputs, spark)